<a href="https://colab.research.google.com/github/TralAlex/IW/blob/main/2_zadatak_fin_sent_of_Big_Data_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

2. Projektni Zadatak: Razvoj Klasifikatora za Analizu
Sentimenta u PySpark-u

**I**  **Priprema podataka**: sastoji se od
1.Prikupljanja podataka (podaci o finansijskim izvestajima)
 2.Prepocesuiranja teksta (provera nedostajicih vrednosti,ukljanjanje stop reci )
 3.Tokenizacija (stemizacija(svodjenje reci na koren))

In [ ]:
#  Povezivanje Google Drive skladišta sa Google Colab okruženjem.
# Dataset se učitava direktno sa Google Drive-a kako bi bio dostupan za obradu u PySpark okruženju.
from google.colab import drive
drive.mount("/content/gdrive")

dataset_path = "/content/gdrive/MyDrive/2 zadatak big data/data.csv"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
# Kreiranje SparkSession objekta koji predstavlja ulaznu tačku za rad sa PySpark-om.
# Spark sesija omogućava obradu velikih skupova podataka i izvršavanje distribuiranih operacija.
from pyspark.sql import SparkSession

sc = SparkSession.builder.appName("FinancialSentimentAnalysis").getOrCreate()


In [ ]:
# Učitavanje CSV dataset-a u Spark DataFrame strukturu.
# Parametri header i inferSchema omogućavaju automatsko prepoznavanje naziva kolona i tipova podataka.
# Prikaz šeme i prvih redova služi za proveru ispravnosti učitanih podataka.
data = sc.read.csv(dataset_path, header=True, inferSchema=True, quote='"', escape='"')
data.printSchema()
data.show()

root
 |-- Sentence: string (nullable = true)
 |-- Sentiment: string (nullable = true)

+--------------------+---------+
|            Sentence|Sentiment|
+--------------------+---------+
|The GeoSolutions ...| positive|
|$ESI on lows, dow...| negative|
|For the last quar...| positive|
|According to the ...|  neutral|
|The Swedish buyou...|  neutral|
|$SPY wouldn't be ...| positive|
|Shell's $70 Billi...| negative|
|SSH COMMUNICATION...| negative|
|Kone 's net sales...| positive|
|The Stockmann dep...|  neutral|
|Circulation reven...| positive|
|$SAP Q1 disappoin...| negative|
|The subdivision m...| positive|
|Viking Line has c...|  neutral|
|Ahlstrom Corporat...|  neutral|
|$FB gone green on...| positive|
|$MSFT SQL Server ...| positive|
|According to L+ñn...|  neutral|
|The company 's sh...|  neutral|
|Elcoteq SE is lis...|  neutral|
+--------------------+---------+
only showing top 20 rows


In [ ]:
# Provera prisustva nedostajućih vrednosti u svim kolonama dataset-a.
# Analiza missing vrednosti je važna kako bi se obezbedio kvalitet podataka pre treniranja modela.
from pyspark.sql.functions import count, when, col

print("Missing values:")
data.select([count(when(col(c).isNull(), c)).alias(c) for c in data.columns]).show()

Missing values:
+--------+---------+
|Sentence|Sentiment|
+--------+---------+
|       0|        0|
+--------+---------+



In [ ]:
# Zadržavaju se samo validne klase sentimenta: positive, neutral i negative.
# Nakon filtriranja vrši se ponovna provera raspodele klasa kako bi se potvrdila ispravnost obrade podataka.
valid_labels = ["positive", "neutral", "negative"]

data = data.filter(col("Sentiment").isin(valid_labels))

data.groupBy("Sentiment").count().show()

+---------+-----+
|Sentiment|count|
+---------+-----+
| positive| 1852|
|  neutral| 3130|
| negative|  860|
+---------+-----+



In [ ]:
# Konverzija tekstualnih oznaka sentimenta u numeričke vrednosti korišćenjem StringIndexer transformacije.
# Numeričke labele su neophodne za treniranje klasifikacionih modela u PySpark ML biblioteci.
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label"
)

data = label_indexer.fit(data).transform(data)
data.select("Sentiment", "label").show(10)

+---------+-----+
|Sentiment|label|
+---------+-----+
| positive|  1.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
| negative|  2.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
+---------+-----+
only showing top 10 rows


In [ ]:
# Dataset se deli na trening i test skup u odnosu 80:20.
# Trening skup se koristi za obučavanje modela, dok se test skup koristi za evaluaciju performansi modela na nepoznatim podacima.
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_data.count())
print("Test:", test_data.count())

Train: 4725
Test: 1117


In [ ]:
# Tekstualni podaci se dele na pojedinačne tokene (reči) korišćenjem Tokenizer transformacije.
# Nakon tokenizacije uklanjaju se stop reči koje nemaju značajan uticaj na analizu sentimenta.
from pyspark.ml.feature import Tokenizer, StopWordsRemover

tokenizer = Tokenizer(
    inputCol="Sentence",
    outputCol="tokens"
)

stop_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

In [ ]:
# Definiše se PorterStemmer za stemizaciju tokena, odnosno svođenje reči na njihov osnovni oblik.
# UDF funkcija omogućava primenu stemizacije nad Spark kolonama koje sadrže nizove tokena.
from nltk.stem import PorterStemmer
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

stem_udf = udf(stem_tokens, ArrayType(StringType()))

In [ ]:
# Funkcija preprocess_data objedinjuje osnovne korake pripreme teksta: tokenizaciju, uklanjanje stop reči i stemizaciju.
# Na ovaj način se dobija očišćen tekstualni ulaz koji je pogodniji za kasniju ekstrakciju karakteristika i treniranje modela.def preprocess_data(df):
    tokenized = tokenizer.transform(df)
    filtered = stop_remover.transform(tokenized)

    stemmed = filtered.withColumn(
        "stemmed_tokens",
        stem_udf(col("filtered_tokens"))
    )

    return stemmed

**II** - **Kreiranje i optimizacija modela** : 1. Kreiranje modela 2. Evaluacija modela.
Koriscene metrike

Accuracy – procenat ukupno tačno klasifikovanih primera.

Precision – koliko su pozitivne predikcije modela zaista tačne.

Recall – koliko dobro model pronalazi stvarno pozitivne primere.

F1-score – balans između Precision i Recall metrike.

In [ ]:
# Funkcija evaluate_model računa glavne metrike za višeklasnu klasifikaciju: tačnost, F1 skor, ponderisanu preciznost i ponderisani odziv.
# Ove metrike omogućavaju objektivno poređenje performansi različitih modela nad test skupom.
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate_model(predictions):
    metrics = {}

    for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        metrics[metric] = evaluator.evaluate(predictions)

    return metrics

In [ ]:
# Funkcija get_model omogućava dinamičko kreiranje različitih klasifikacionih modela.
# Implementirani modeli uključuju logističku regresiju, stablo odlučivanja i Random Forest ansambl algoritam.
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

def get_model(model_name, params=None):
    params = params or {}

    if model_name == "lr":
        return LogisticRegression(featuresCol="features", labelCol="label", **params)

    elif model_name == "dt":
        return DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42, **params)

    elif model_name == "rf":
        return RandomForestClassifier(featuresCol="features", labelCol="label", seed=42, **params)

    else:
        raise ValueError("Unknown model")

TF-IDF je metoda za određivanje važnosti reči u dokumentu u odnosu na ceo skup dokumenata. Kombinuje učestalost reči u dokumentu (TF) i retkost reči u svim dokumentima (IDF).
**TFIDF(t,d,D)=TF(t,d)⋅IDF(t,D)**

Visok TF-IDF znači da je reč značajna za taj dokument, dok nizak TF-IDF označava česte i manje informativne reči.

In [ ]:
# Tekstualni podaci se transformišu u numeričke karakteristike korišćenjem TF-IDF metode.
# CountVectorizer generiše matricu frekvencija termina, dok IDF umanjuje značaj često prisutnih reči u dokumentima.
from pyspark.ml.feature import CountVectorizer, IDF

def build_tfidf_features(train_df, test_df):
    cv = CountVectorizer(
        inputCol="stemmed_tokens",
        outputCol="raw_features",
        vocabSize=5000,
        minDF=5
    )
    cv_model = cv.fit(train_df)

    train_featurized = cv_model.transform(train_df)
    test_featurized = cv_model.transform(test_df)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

In [ ]:
# Kreiranje bigrama omogućava modelu da analizira kombinacije dve uzastopne reči umesto pojedinačnih termina.
# Nakon generisanja bigrama primenjuje se TF-IDF transformacija radi dobijanja numeričkih reprezentacija teksta.
from pyspark.ml.feature import NGram

def build_bigram_features(train_df, test_df):
    ngram = NGram(
        n=2,
        inputCol="stemmed_tokens",
        outputCol="bigrams"
    )

    train_ngram = ngram.transform(train_df)
    test_ngram = ngram.transform(test_df)

    cv = CountVectorizer(
        inputCol="bigrams",
        outputCol="raw_features",
        vocabSize=5000,
        minDF=5
    )
    cv_model = cv.fit(train_ngram)

    train_featurized = cv_model.transform(train_ngram)
    test_featurized = cv_model.transform(test_ngram)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

Word2Vec je algoritam koji pretvara reči u vektore (embeddings) tako da reči sa sličnim značenjem imaju slične vektore. Koristi kontekst reči tokom treniranja pomoću CBOW ili Skip-gram pristupa.

Koraci:

-Priprema teksta i pravljenje vokabulara
-Definisanje context window-a
-Treniranje modela predviđanjem reči ili konteksta
-Optimizacija gradient descent metodom
-Učenje word embedding vektora

In [ ]:
# Word2Vec model generiše vektorske reprezentacije reči na osnovu njihovog konteksta u tekstu.
# Na ovaj način se omogućava očuvanje semantičkih odnosa između reči prilikom predstavljanja teksta numeričkim karakteristikama.
from pyspark.ml.feature import Word2Vec

def build_word2vec_features(train_df, test_df):
    word2vec = Word2Vec(
        vectorSize=100,
        minCount=1,
        inputCol="stemmed_tokens",
        outputCol="features"
    )

    w2v_model = word2vec.fit(train_df)

    train_final = w2v_model.transform(train_df)
    test_final = w2v_model.transform(test_df)

    return train_final, test_final

In [ ]:
# Funkcija prepare_features objedinjuje proces pripreme podataka i kreiranja karakteristika.
# U zavisnosti od izabrane metode moguće je koristiti TF-IDF, TF-IDF sa bigramima ili Word2Vec reprezentaciju teksta.
def prepare_features(train_data, test_data, feature_method):
    train_processed = preprocess_data(train_data)
    test_processed = preprocess_data(test_data)

    if feature_method == "tfidf":
        return build_tfidf_features(train_processed, test_processed)

    elif feature_method == "bigram_tfidf":
        return build_bigram_features(train_processed, test_processed)

    elif feature_method == "word2vec":
        return build_word2vec_features(train_processed, test_processed)

    else:
        raise ValueError("Unknown feature method")

In [ ]:
# Funkcija train_and_evaluate vrši treniranje klasifikacionog modela nad pripremljenim podacima.
# Nakon treniranja model se evaluira nad test skupom korišćenjem definisanih metrika performansi.
def train_and_evaluate(train_final, test_final, model_name, params=None):
    model = get_model(model_name, params)
    trained_model = model.fit(train_final)

    predictions = trained_model.transform(test_final)

    metrics = evaluate_model(predictions)

    return {
        "model": trained_model,
        "metrics": metrics
    }

In [ ]:
# Rečnik param_grid sadrži različite kombinacije hiperparametara za svaki klasifikacioni model.
# Na ovaj način se omogućava automatsko testiranje više konfiguracija logističke regresije, stabla odlučivanja i Random Forest modela.
param_grid = {
    "lr": [
        {"maxIter": 100, "regParam": 0.01, "elasticNetParam": 0.0},
        {"maxIter": 100, "regParam": 0.1,  "elasticNetParam": 0.0},
        {"maxIter": 100, "regParam": 0.01, "elasticNetParam": 0.5},
        {"maxIter": 100, "regParam": 0.1,  "elasticNetParam": 0.5},
        {"maxIter": 100, "regParam": 0.01, "elasticNetParam": 1.0},
    ],

    "dt": [
        {"maxDepth": 8,  "minInstancesPerNode": 2},
        {"maxDepth": 10, "minInstancesPerNode": 2},
        {"maxDepth": 12, "minInstancesPerNode": 5},
        {"maxDepth": 15, "minInstancesPerNode": 10},
    ],

    "rf": [
        {"numTrees": 100, "maxDepth": 10, "featureSubsetStrategy": "sqrt"},
        {"numTrees": 150, "maxDepth": 12, "featureSubsetStrategy": "sqrt"},
    ]
}

In [ ]:
# Funkcija run_all_experiments automatski prolazi kroz sve kombinacije metoda za obradu teksta, modela i hiperparametara.
# Pripremljeni skupovi podataka se keširaju kako bi se ubrzalo višestruko treniranje modela nad istim karakteristikama.
# Kao izlaz se dobija rečnik sa metrikama performansi za svaku testiranu kombinaciju.
def run_all_experiments(train_data, test_data, param_grid):
    results = {}

    feature_methods = ["tfidf", "bigram_tfidf", "word2vec"]
    prepared_data = {}

    for feature_method in feature_methods:
        print(f"Preparing {feature_method} features...")
        train_final, test_final = prepare_features(train_data, test_data, feature_method)
        # cache
        train_final = train_final.cache()
        test_final = test_final.cache()
        # force computation
        train_final.count()
        test_final.count()
        prepared_data[feature_method] = (train_final, test_final)

    for feature_method, (train_final, test_final) in prepared_data.items():
        for model_name, param_list in param_grid.items():
            for params in param_list:
                key = f"{feature_method}_{model_name}_{params}"

                print(f"Running {key}...")

                result = train_and_evaluate(
                    train_final,
                    test_final,
                    model_name,
                    params
                )

                results[key] = result["metrics"]

    return results

In [ ]:
# Pokretanje kompletne procedure optimizacije nad svim kombinacijama metoda obrade teksta, modela i hiperparametara.
# Rezultati svih eksperimenata čuvaju se u rečniku all_results radi kasnije analize i poređenja performansi.

all_results = run_all_experiments(train_data, test_data, param_grid)

Preparing tfidf features...
Preparing bigram_tfidf features...
Preparing word2vec features...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.0}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'elasticNetParam': 0.0}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.5}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'elasticNetParam': 0.5}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 1.0}...
Running tfidf_dt_{'maxDepth': 8, 'minInstancesPerNode': 2}...
Running tfidf_dt_{'maxDepth': 10, 'minInstancesPerNode': 2}...
Running tfidf_dt_{'maxDepth': 12, 'minInstancesPerNode': 5}...
Running tfidf_dt_{'maxDepth': 15, 'minInstancesPerNode': 10}...
Running tfidf_rf_{'numTrees': 100, 'maxDepth': 10, 'featureSubsetStrategy': 'sqrt'}...
Running tfidf_rf_{'numTrees': 150, 'maxDepth': 12, 'featureSubsetStrategy': 'sqrt'}...
Running bigram_tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.0}...
R

- Evaluacija modela i interpretacija rezultata

In [ ]:
# Izbor najbolje konfiguracije modela na osnovu najveće vrednosti F1 metrike.
# F1 skor predstavlja balans između preciznosti i odziva i pogodan je za evaluaciju višeklasnih klasifikacionih problema.

best_model = max(all_results.items(), key=lambda x: x[1]["f1"])

print("Best configuration:")
print(best_model[0])
print(best_model[1])

Best configuration:
tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 1.0}
{'accuracy': 0.738585496866607, 'f1': 0.7105387712783952, 'weightedPrecision': 0.7185870420675095, 'weightedRecall': 0.7385854968666069}


In [ ]:
# Rezultati svih eksperimenata konvertuju se u Pandas DataFrame radi lakšeg poređenja modela i vizuelne analize performansi.
# Tabela se sortira opadajuće prema F1 metrici kako bi najbolji modeli bili prikazani na vrhu.
import pandas as pd

results_table = []

for name, metrics in all_results.items():
    row = {"experiment": name}
    row.update(metrics)
    results_table.append(row)

results_df = pd.DataFrame(results_table)
results_df.sort_values(by="f1", ascending=False)

,experiment,accuracy,f1,weightedPrecision,weightedRecall
4,"tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'e...",0.738585,0.710539,0.718587,0.738585
2,"tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'e...",0.729633,0.710217,0.702699,0.729633
1,"tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'el...",0.697404,0.681320,0.670943,0.697404
6,"tfidf_dt_{'maxDepth': 10, 'minInstancesPerNode...",0.712623,0.663808,0.687751,0.712623
7,"tfidf_dt_{'maxDepth': 12, 'minInstancesPerNode...",0.702775,0.660053,0.664656,0.702775
5,"tfidf_dt_{'maxDepth': 8, 'minInstancesPerNode'...",0.709937,0.659342,0.692335,0.709937
8,"tfidf_dt_{'maxDepth': 15, 'minInstancesPerNode...",0.692032,0.657509,0.652595,0.692032
0,"tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'e...",0.649060,0.646922,0.644859,0.649060
3,"tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'el...",0.670546,0.601237,0.565562,0.670546
22,"word2vec_lr_{'maxIter': 100, 'regParam': 0.01,...",0.638317,0.594517,0.579614,0.638317


Interpretacija rezultata

Najbolji model na Financial Sentiment Analysis datasetu je Logistic Regression sa TF-IDF reprezentacijom, koji je ostvario najbolje rezultate (Accuracy ≈ 0.739, F1 ≈ 0.711).

TF-IDF se pokazao boljim od Word2Vec i bigrama, što znači da su ključne pojedinačne reči dovoljne za dobru klasifikaciju sentimenta.

Decision Tree je dao solidne, ali slabije rezultate, dok Random Forest i bigram pristupi imaju najlošije performanse zbog visoke dimenzionalnosti i šuma.

Zaključak

Najefikasniji pristup je TF-IDF + Logistic Regression, jer daje najbolji balans tačnosti i stabilnosti, što potvrđuje da su obrasci u podacima uglavnom linearno separabilni.

In [ ]:
# Dodatna interpretacija rezultata:
# Logistic Regression postiže najbolje rezultate jer TF-IDF reprezentacija generiše visokodimenzionalne,
# ali linearno separabilne karakteristike, što odgovara prirodi logističke regresije.
#
# Word2Vec pristup omogućava semantičko predstavljanje reči, ali na ovom dataset-u nije uspeo
# da nadmaši TF-IDF zbog relativno kratkih finansijskih tekstova.
#
# Bigram modeli uvode dodatni kontekst između susednih reči, ali povećavaju dimenzionalnost
# prostora karakteristika, što može dovesti do šuma i slabije generalizacije modela.
#
# Random Forest i Decision Tree modeli pokazuju stabilne rezultate, ali su manje efikasni
# u radu sa sparse tekstualnim reprezentacijama u odnosu na logističku regresiju.